<a href="https://colab.research.google.com/github/YoussefAli07/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/YoussefAli07/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 335, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 335 (delta 146), reused 90 (delta 90), pack-reused 144 (from 1)
Receiving objects: 100% (335/335), 1.92 MiB | 4.71 MiB/s, done.
Resolving deltas: 100% (187/187), done.


In [6]:
import pandas as pd
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "High search volume is a weak expectation-setting number"
**Claim:** The paper reports a near-zero correlation between search volume and impressions
(raw r = 0.0083, log-scaled r = -0.0419), concluding that search volume weakly predicts traffic.

**Methodology question:** This correlation is computed only on the "local active-content sample,"
which is filtered to `impressions_90d > 0 AND sessions_90d > 0`. That filter excludes, by
construction, any high-search-volume page that earned *zero* traffic — which is exactly the
failure mode a reader would most want this finding to speak to. The claim as tested is closer to
"among pages that already got some traffic, search volume doesn't predict how much more" — not
"search volume doesn't predict whether a page gets traffic at all." Was the zero-traffic
population checked separately before generalizing the claim to the whole portfolio?

### Finding 2: "Click capture by position tier"
**Claim:** Weighted CTR falls sharply as position tier moves away from the top of search results
— roughly an 88% drop from Top 3 (0.423%) to Deep (0.050%).

**Methodology question:** The paper doesn't disclose whether rows with `avg_position = 0` —
which represent *missing* position data, not an actual top rank — were excluded before pages
were bucketed into tiers. If any such rows landed in a tier (most plausibly "Deep," since 0
would sort as the lowest numeric value), their clicks/impressions would distort that tier's
weighted CTR without reflecting real ranking behavior at all. Confirming this exclusion happened
would strengthen an otherwise well-aggregated (weighted-CTR-over-per-row-average) methodology.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
# Same feature set and target as ML-08 — kept identical so any MAE
# difference below comes from the SPLIT METHOD, not from different inputs.
feature_cols = ['search_volume', 'competition', 'content_type', 'main_intent', 'word_count']
X = df[feature_cols]
y = df['trend_pct']
X = pd.get_dummies(X, columns=['content_type', 'main_intent'])

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# HONEST split: grouped by client_id, same as ML-08
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

# Drop missing target rows
valid_train_g = y_train_g.notnull()
X_train_g, y_train_g = X_train_g[valid_train_g], y_train_g[valid_train_g]
valid_test_g = y_test_g.notnull()
X_test_g, y_test_g = X_test_g[valid_test_g], y_test_g[valid_test_g]

# Cap training target at 1st/99th percentile (train only, same as ML-08) — for the MODEL only
lower_cap_g = y_train_g.quantile(0.01)
upper_cap_g = y_train_g.quantile(0.99)
y_train_g_capped = y_train_g.clip(lower=lower_cap_g, upper=upper_cap_g)

# Naive baseline uses the RAW, uncapped y_train mean — a naive guesser
# wouldn't cap outliers first, so this is the fair comparison point.
naive_pred_g = y_train_g.mean()
naive_mae_g = mean_absolute_error(y_test_g, np.full(len(y_test_g), naive_pred_g))

# Model trains on the capped target (legitimate, disclosed modeling choice)
tree_model_g = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_model_g.fit(X_train_g, y_train_g_capped)
tree_mae_g = mean_absolute_error(y_test_g, tree_model_g.predict(X_test_g))

print("=== HONEST split (grouped by client_id) ===")
print("Naive baseline MAE (uncapped mean):", naive_mae_g)
print("Decision Tree MAE (capped training):", tree_mae_g)
print("Beats naive by:                     ", naive_mae_g - tree_mae_g)

=== HONEST split (grouped by client_id) ===
Naive baseline MAE (uncapped mean): 68.14762960803775
Decision Tree MAE (capped training): 65.24051816119345
Beats naive by:                      2.907111446844297


In [13]:
from sklearn.model_selection import train_test_split

# DISHONEST split: random, no client grouping — rows from the same
# client can end up in both train and test.
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Drop missing target rows
valid_train_r = y_train_r.notnull()
X_train_r, y_train_r = X_train_r[valid_train_r], y_train_r[valid_train_r]
valid_test_r = y_test_r.notnull()
X_test_r, y_test_r = X_test_r[valid_test_r], y_test_r[valid_test_r]

# Cap training target at 1st/99th percentile (train only, same rule as honest split) — for the MODEL only
lower_cap_r = y_train_r.quantile(0.01)
upper_cap_r = y_train_r.quantile(0.99)
y_train_r_capped = y_train_r.clip(lower=lower_cap_r, upper=upper_cap_r)

# Naive baseline uses the RAW, uncapped y_train mean — same rule as the honest split
naive_pred_r = y_train_r.mean()
naive_mae_r = mean_absolute_error(y_test_r, np.full(len(y_test_r), naive_pred_r))

# Model trains on the capped target (legitimate, disclosed modeling choice)
tree_model_r = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_model_r.fit(X_train_r, y_train_r_capped)
tree_mae_r = mean_absolute_error(y_test_r, tree_model_r.predict(X_test_r))

print("=== DISHONEST split (random, no client grouping) ===")
print("Naive baseline MAE (uncapped mean):", naive_mae_r)
print("Decision Tree MAE (capped training):", tree_mae_r)
print("Beats naive by:                     ", naive_mae_r - tree_mae_r)

=== DISHONEST split (random, no client grouping) ===
Naive baseline MAE (uncapped mean): 69.44322307782362
Decision Tree MAE (capped training): 63.45631108412796
Beats naive by:                      5.9869119936956565


In [14]:
print(f"{'Split type':<25}{'Naive MAE':>12}{'Model MAE':>12}{'Beats naive by':>18}")
print(f"{'Honest (grouped)':<25}{naive_mae_g:>12.2f}{tree_mae_g:>12.2f}{naive_mae_g - tree_mae_g:>18.2f}")
print(f"{'Dishonest (random)':<25}{naive_mae_r:>12.2f}{tree_mae_r:>12.2f}{naive_mae_r - tree_mae_r:>18.2f}")

Split type                  Naive MAE   Model MAE    Beats naive by
Honest (grouped)                68.15       65.24              2.91
Dishonest (random)              69.44       63.46              5.99


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.